# FreeAlexya sur GPU gratuit (Colab T4)
Menu **Exécution > Modifier le type d'exécution > GPU T4**, puis lancez les cellules dans l'ordre.

In [ ]:
!nvidia-smi -L
!git clone -q https://github.com/comfyanonymous/ComfyUI /content/ComfyUI
!pip install -q -r /content/ComfyUI/requirements.txt "mcp[cli]" requests
!wget -q -O /content/ComfyUI/models/checkpoints/sd_xl_turbo_1.0_fp16.safetensors https://huggingface.co/stabilityai/sdxl-turbo/resolve/main/sd_xl_turbo_1.0_fp16.safetensors
!wget -q -O /content/ComfyUI/models/checkpoints/ltx-video-2b-v0.9.5.safetensors https://huggingface.co/Lightricks/LTX-Video/resolve/main/ltx-video-2b-v0.9.5.safetensors
!wget -q -O /content/ComfyUI/models/text_encoders/t5xxl_fp8_e4m3fn_scaled.safetensors https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn_scaled.safetensors
# --- remplacement de personnage (Wan VACE 1.3B + extraction de pose) ---
!git clone -q https://github.com/Fannovel16/comfyui_controlnet_aux /content/ComfyUI/custom_nodes/comfyui_controlnet_aux
!pip install -q -r /content/ComfyUI/custom_nodes/comfyui_controlnet_aux/requirements.txt
!wget -q -O /content/ComfyUI/models/diffusion_models/wan2.1_vace_1.3B_fp16.safetensors https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/diffusion_models/wan2.1_vace_1.3B_fp16.safetensors
!wget -q -O /content/ComfyUI/models/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors
!wget -q -O /content/ComfyUI/models/vae/wan_2.1_vae.safetensors https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared

## Envoyez uniquement `server.py` (la vidéo LTX est déjà intégrée)

In [ ]:
from google.colab import files
import os
os.makedirs("/content/app/workflows", exist_ok=True)
%cd /content/app
for name, data in files.upload().items():
    dest = "workflows/" + name if name.endswith(".json") else name
    open(dest, "wb").write(data)

In [ ]:
import subprocess, time, re, os
subprocess.Popen("python main.py --listen 127.0.0.1 --port 8188", shell=True, cwd="/content/ComfyUI",
                 stdout=open("/content/comfy.log","w"), stderr=subprocess.STDOUT)
tun = subprocess.Popen("cloudflared tunnel --url http://localhost:8000", shell=True,
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tun.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m: url = m.group(0); break
print("Connecteur Claude :", url + "/mcp")
os.environ.update(PUBLIC_URL=url, MODEL="sdxl_turbo", COMFY_URL="http://127.0.0.1:8188")
time.sleep(20)  # laisser ComfyUI démarrer

In [ ]:
!python server.py